*   Mesh generator -> generate_mu
*   Cambios en la seed design_matrices
*   Cambios en dictlayer con el psi
*   SESMS: Como tal
*   sub_block_partition: generate_list_of_subblock
*   Dictlayer como tal


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!mkdir results_1
!mkdir results_2
!mkdir results_3
!mkdir results_1/plots
!mkdir results_2/plots
!mkdir results_3/plots
!mkdir results_1/stats
!mkdir results_2/stats
!mkdir results_3/stats

mkdir: cannot create directory ‘results_1’: File exists
mkdir: cannot create directory ‘results_2’: File exists
mkdir: cannot create directory ‘results_3’: File exists
mkdir: cannot create directory ‘results_1/plots’: File exists
mkdir: cannot create directory ‘results_2/plots’: File exists
mkdir: cannot create directory ‘results_3/plots’: File exists
mkdir: cannot create directory ‘results_1/stats’: File exists
mkdir: cannot create directory ‘results_2/stats’: File exists
mkdir: cannot create directory ‘results_3/stats’: File exists


In [3]:
import sys
import torch
import plotly.graph_objects as go
import pandas as pd
sys.path.append('/content/drive/MyDrive/SESM')

from pysesm.utils.gaussian_covariance_density import *
from pysesm.utils.mesh_generation import *
from pysesm.utils.loggers import *
from pysesm.utils.design_matrices import *
from pysesm.models.SSESM.SESMS import *
from pysesm.models.SESM.SESM import *
from pysesm.base_functions.sub_block_partition import *
from pysesm.functions import *
from pysesm.functions import *
from pysesm.plotting.sesm_plotting import *
from pysesm.utils.plot_and_save_stats import *


In [4]:
N_iter = 1 #
experiment_1 = {
      "hyp_set": 1,
      "n_samples"	: 500,
      "n_features" : 2,
      "l_functions":  100,
      "eig_range": [1e0, 1e1],
      "mu_range": [-2, 2],
      "vector_range": [1e0, 1e1],
      "ista_alpha"	: 0.05502,
      "ista_lambd"	 : 0.01007,
      "dictionary_alpha": 0.08928,
      "rho_epochs": 10,
      "mu_epochs": 10,
      "model_epochs" : 2,
      "dict_epochs": 2,
      "ista_epochs": 1,
      "T": 4,
      "weight_decay": 0.004875,
      "permutation_times": 1,
      "mode": "secuencial",
      "Seed": 45
}


In [5]:
#Logger
logger = setup_logger()
logger.setLevel(logging.INFO)

In [6]:
#Seed
SEED = 45

In [7]:
# Definicion de covarianzas no diagnonales
sigma1, sigma2, sigma3 = generate_sigma_tensors()

In [8]:
#Definicion de varianzas diagonales
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

In [9]:
mu1 = generate_mu(1, 1)
mu2 = generate_mu(1, -1)
mu3 = generate_mu(-1, -1)

#sigmas Non-diagonal covariance
sigma_list = [sigma1_d, sigma2_d, sigma3_d]

mu_list = [mu1,mu2,mu3]

xx, yy, zz = generate_mesh(50, -2, 2, sigma_list, mu_list)


xx_r, yy_r, zz_r = generate_random_samples(500, -2, 2, sigma_list, mu_list, SEED)


In [10]:
fig = go.Figure(data=[go.Surface(z=zz.numpy(), x=xx, y=yy)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

In [11]:


# Instanciar la función de aproximación (surrogate function)
surrogate_function = GaussianApproximateSurrogateFunction(
    n_features=experiment_1["n_features"],
    n_functions=experiment_1["l_functions"],
    logger=logger,
    eig_range=experiment_1["eig_range"],
    mu_range=experiment_1["mu_range"],
    vector_range=experiment_1["vector_range"],
    seed=SEED
)

# Instanciando el modelo SESM con los parámetros del experimento
sesm_model = SESM(
    n_samples=experiment_1["n_samples"],
    psi=surrogate_function,
    seed=SEED,
    model_epochs=experiment_1["model_epochs"],
    ista_epochs=experiment_1["ista_epochs"],
    ista_alpha=experiment_1["ista_alpha"],
    ista_lambd=experiment_1["ista_lambd"],
    mu_epochs=experiment_1["mu_epochs"],
    rho_epochs=experiment_1["rho_epochs"],
    dictionary_alpha=experiment_1["dictionary_alpha"],
    weight_decay=experiment_1["weight_decay"]
)

# Crea una instancia de la clase SESMS
sesms_model = SESMS(
    n_samples=experiment_1["n_samples"],
    n_features=experiment_1["n_features"],
    l_functions=experiment_1["l_functions"],
    eig_range=tuple(experiment_1["eig_range"]),
    mu_range=tuple(experiment_1["mu_range"]),
    vector_range=tuple(experiment_1["vector_range"]),
    model_epochs=experiment_1["model_epochs"],
    ista_epochs=experiment_1["ista_epochs"],
    rho_epochs=experiment_1["rho_epochs"],
    mu_epochs=experiment_1["mu_epochs"],
    ista_alpha=experiment_1["ista_alpha"],
    ista_lambd=experiment_1["ista_lambd"],
    dictionary_alpha=experiment_1["dictionary_alpha"],
    weight_decay=experiment_1["weight_decay"],
    surrogate_function=surrogate_function,
    permutation_times=experiment_1["permutation_times"],
    dfngroup=1,
    iter=0,
    seed=SEED,
    logger=logger,
    T=experiment_1["T"],
    debug=True
)


INFO:logger:Shape of Mu: torch.Size([2, 100])
INFO:logger:Shape of Rho: torch.Size([3, 100])
INFO:logger:Shape of Theta: torch.Size([5, 100])
INFO:logger:Shape of Mu: torch.Size([2, 100])
INFO:logger:Shape of Rho: torch.Size([3, 100])
INFO:logger:Shape of Theta: torch.Size([5, 100])


In [12]:
# Dataset
data = []
trainDataset = {"X": xx_r.ravel(), "Y": yy_r.ravel(), "Z": zz_r.ravel() }
testDataset = {"X": xx.ravel(), "Y": yy.ravel(), "Z": zz.ravel() }
# Crear la matriz de diseño
X_train, y_train = create_design_matrix_train(xx_r, yy_r, zz_r, experiment_1)
# Crear la matriz de diseño
X_test, y_test = create_design_matrix_test(xx, yy, zz)


/content/drive/MyDrive/SESM/pysesm/utils/sampling_data.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(z_values[sampled_indices], dtype=torch.float32)


In [17]:
# Ejecutar el experimento
# Cambiar los datos por una matriz de matriz de diseño X
# Un condicional para una grafica de > dimensiones
# run_experiment deberia de estar fuera de esta clase

def run_experiment(X_train, y_train, X_test, y_test, hyperparams, model):

  T = hyperparams["T"]
  l_functions = hyperparams["l_functions"]
  seed = hyperparams["Seed"]
  weight_decay = hyperparams["weight_decay"]
  alpha = hyperparams["ista_alpha"]
  lambd = hyperparams["ista_lambd"]

  t, x_n = data_mapping(X_train, T)
  count_unique_combinations(t)

  sub_blocks = locate_samples_in_sub_blocks(x_n, y_train, t, T)

  list_sub_blocks = generate_list_of_subblock(sub_blocks,l_functions, seed, weight_decay, alpha,lambd)

  time, mse_value = 0, 0

  model.partial_fit(list_sub_blocks, T)
  Z, time, mse_value = model.performance_stats(X_test, y_test, list_sub_blocks)
  plot_surface(testDataset, X_train, y_train, Z, experiment_1["hyp_set"], model.dfngroup, model.iter,model.ista_layer_losses, model.dictionary_layer_losses)

  # df = pd.DataFrame({
  #           'Loss_ISTA':  model.ista_layer_losses[:min_len],
  #           'Loss_Dictionary': model.dictionary_layer_losses[:min_len]
  #       })
  # df.to_csv(f'results_{experiment_1["hyp_set"]}/stats/{model.iter}.csv', index=False)

  return time, mse_value

In [18]:
for i in range(N_iter):

  sesms_model.iter = i
  #Ejecuta el experimento

  time, mse = run_experiment(X_train, y_train, X_test, y_test, experiment_1, sesms_model)

  # Almacena los resultados
  data.append((i, time, mse))

save_results(data=data, fngroup=1)

Combinación (3, 2): 34 veces
Combinación (0, 2): 24 veces
Combinación (2, 0): 32 veces
Combinación (1, 1): 36 veces
Combinación (2, 1): 18 veces
Combinación (0, 3): 30 veces
Combinación (1, 3): 32 veces
Combinación (2, 2): 36 veces
Combinación (3, 3): 23 veces
Combinación (3, 1): 34 veces
Combinación (1, 0): 38 veces
Combinación (0, 0): 37 veces
Combinación (0, 1): 27 veces
Combinación (1, 2): 31 veces
Combinación (2, 3): 28 veces
Combinación (3, 0): 40 veces
OUTPUT VALUES: 37
OUTPUT VALUES: 27
OUTPUT VALUES: 24
OUTPUT VALUES: 30
OUTPUT VALUES: 38
OUTPUT VALUES: 36
OUTPUT VALUES: 31
OUTPUT VALUES: 32
OUTPUT VALUES: 32
OUTPUT VALUES: 18
OUTPUT VALUES: 36
OUTPUT VALUES: 28
OUTPUT VALUES: 40
OUTPUT VALUES: 34
OUTPUT VALUES: 34
OUTPUT VALUES: 23


/content/drive/MyDrive/SESM/pysesm/base_functions/sub_block_partition.py:156: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_sub_block = torch.tensor(sub_block_points, dtype=torch.float32)


CURRENT ISTA on sub block  Parameter containing:
tensor([0.0031, 0.0199, 0.0136, 0.0182, 0.0000, 0.0108, 0.0158, 0.0001, 0.0166,
        0.0021, 0.0000, 0.0029, 0.0081, 0.0040, 0.0163, 0.0104, 0.0143, 0.0129,
        0.0121, 0.0200, 0.0030, 0.0015, 0.0080, 0.0199, 0.0123, 0.0264, 0.0077,
        0.0064, 0.0057, 0.0150, 0.0072, 0.0238, 0.0173, 0.0159, 0.0118, 0.0010,
        0.0155, 0.0067, 0.0023, 0.0114, 0.0111, 0.0008, 0.0174, 0.0028, 0.0137,
        0.0021, 0.0174, 0.0153, 0.0022, 0.0130, 0.0060, 0.0064, 0.0125, 0.0132,
        0.0008, 0.0185, 0.0058, 0.0103, 0.0112, 0.0170, 0.0167, 0.0149, 0.0054,
        0.0041, 0.0173, 0.0064, 0.0019, 0.0269, 0.0123, 0.0110, -0.0000, 0.0151,
        0.0028, 0.0181, 0.0156, 0.0186, 0.0096, 0.0230, 0.0063, 0.0073, -0.0000,
        0.0118, 0.0070, 0.0252, 0.0073, 0.0190, 0.0123, 0.0025, 0.0010, 0.0018,
        0.0016, 0.0108, 0.0245, 0.0069, 0.0041, 0.0108, 0.0013, 0.0063, 0.0020,
        0.0169], requires_grad=True)
SUM VALUES H  tensor(1.0239)
CUR